# 第7章 智能语音与语言处理

本课程对应《第7章 - 智能嵌入式语音和语言处理》，将带你系统了解智能语音系统的核心技术体系，并在昇腾 NPU 平台上亲手实践语音与语言处理的关键环节。

**运行环境**：`cann_9.0.0-py3.11-A2-arm` · `ASCEND 1*NPU 910B3` · `16vCPUs, 32GiB`

你将学到：
1. 智能语音系统的整体架构与三大核心环节
2. 语音采集与前处理技术（MFCC、VAD、降噪）
3. 语音识别（ASR）与语音合成（TTS）原理
4. 自然语言理解（NLU）与意图识别
5. 大语言模型（LLM）在昇腾平台上的部署实践

---

## 1. 智能语音系统概述

智能语音系统是一个**信号与信息处理流水线**，通过端到端协同工作，实现机器与人类自然语音的无缝交互。

### 1.1 三大核心环节

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">环节</th>
<th style="text-align: left;">核心任务</th>
<th style="text-align: left;">关键技术</th>
</tr>
<tr>
<td style="text-align: left;">① 语音采集与前处理</td>
<td style="text-align: left;">将声波高质量地转换为数字信号</td>
<td style="text-align: left;">麦克风阵列、ADC、降噪、VAD、AEC</td>
</tr>
<tr>
<td style="text-align: left;">② 语音识别与合成</td>
<td style="text-align: left;">实现语音 ⇄ 文本双向转换</td>
<td style="text-align: left;">MFCC、KWS、ASR、TTS</td>
</tr>
<tr>
<td style="text-align: left;">③ 自然语言理解</td>
<td style="text-align: left;">理解用户意图并智能响应</td>
<td style="text-align: left;">意图识别、槽位填充、LLM</td>
</tr>
</table>

**表格解读**：上表将智能语音系统拆解为三个递进的核心环节。第一环节"语音采集与前处理"是整个系统的入口，负责将物理世界的声波转换为计算机可处理的数字信号，关键技术包括麦克风阵列（多通道空间信息采集）、ADC（模数转换）、降噪（去除环境噪声）、VAD（语音活性检测，判断何时有人说话）和AEC（回声消除，去除扬声器回声）。第二环节"语音识别与合成"实现语音与文本的双向桥梁，ASR将语音转为文本，TTS将文本转为语音，MFCC是其中最经典的特征提取方法。第三环节"自然语言理解"是系统的"大脑"，负责理解用户意图并生成智能响应，从早期的规则匹配发展到如今的大语言模型（LLM）。三个环节层层递进，前一个环节的输出是后一个环节的输入，任何一个环节的质量下降都会影响最终用户体验。

端到端工作流程：

```
语音采集 → 前处理 → 特征提取 → 语音识别 → 语义理解 → 响应生成 → 语音合成 → 音频输出
```

**流程说明**：上述流程展示了从用户开口说话到设备输出语音回复的完整数据通路。语音采集阶段通过麦克风获取模拟声波；前处理阶段进行降噪和VAD裁剪；特征提取阶段将时域波形转换为MFCC等特征向量；语音识别阶段将特征解码为文本；语义理解阶段解析用户意图；响应生成阶段组织回复内容；语音合成阶段将文本转换为语音波形；最终通过扬声器输出。在端侧设备上，整个流程需要在数百毫秒内完成，才能给用户即时响应的体验。

### 1.2 典型应用场景

- **智能家居**：语音控制家电、智能场景联动
- **可穿戴设备**：语音助手、健康监测、实时翻译
- **车载系统**：语音导航、车载娱乐、驾驶辅助

> 智能语音市场规模持续增长，预计 2026 年全球市场规模将达到 270 亿美元，年复合增长率超过 20%。边缘 AI 的兴起推动语音技术向**本地化、低延迟、高隐私**方向发展。

## 2. 语音采集与前处理技术

### 2.1 语音采集方案对比

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方案</th>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">局限</th>
</tr>
<tr>
<td style="text-align: left;">单麦克风</td>
<td style="text-align: left;">成本低、结构简单、功耗低</td>
<td style="text-align: left;">嘈杂环境易受干扰、无法定位声源</td>
</tr>
<tr>
<td style="text-align: left;">麦克风阵列</td>
<td style="text-align: left;">波束聚焦、声源定位、强降噪</td>
<td style="text-align: left;">成本较高、算法复杂</td>
</tr>
</table>

**表格解读**：单麦克风方案成本低、结构简单、功耗低，适合对成本敏感的入门级设备（如智能音箱初代产品），但在嘈杂环境中容易受到环境噪声干扰，且无法定位声源方向。麦克风阵列方案通过多个麦克风的空间信息，实现波束聚焦（增强目标方向语音）、声源定位（判断说话人位置）和强降噪（空间滤波），适合车载、会议室等复杂声学场景，代价是成本较高且需要复杂的波束成形算法。在实际产品中，通常采用2~6个麦克风组成的阵列，配合DSP芯片实时处理。

### 2.2 音频前处理流程

```
模拟信号 → ADC转换 → 音频编解码 → 信号优化(降噪/VAD/AEC) → 干净音频
```

**流程说明**：模拟信号经ADC（模数转换器）采样为数字信号，再通过音频编解码器进行格式转换和压缩，最后经降噪（去除背景噪声）、VAD（裁剪静音段）和AEC（消除回声）等信号优化处理，输出干净的音频数据供后续特征提取使用。每一步都直接影响最终识别准确率。

关键参数：
- **采样率**：16 kHz（语音识别）/ 44.1 kHz（音乐）/ 48 kHz（专业）
- **位深**：16 bit / 24 bit
- **声道**：单声道 / 立体声

> **奈奎斯特采样定理**：采样率必须大于信号最高频率的 2 倍，才能无失真还原原信号。人耳可听频率上限约 20 kHz，因此 CD 音质采用 44.1 kHz 采样率。

### 2.3 亲手生成与可视化一段语音信号

下面用 Python 生成一段模拟语音信号（叠加正弦波 + 噪声），并绘制波形，直观感受音频信号是什么样的。

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sample_rate = 16000  # 16 kHz 采样率
duration = 1.0       # 1 秒
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# 模拟语音信号：基频 200Hz + 谐波 400Hz + 随机噪声
signal = (
    0.6 * np.sin(2 * np.pi * 200 * t) +
    0.3 * np.sin(2 * np.pi * 400 * t) +
    0.1 * np.random.randn(len(t))
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 时域波形（前 50ms）
axes[0].plot(t[:800] * 1000, signal[:800])
axes[0].set_title('Time Domain Waveform (first 50ms)')
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')

# Frequency Domain Spectrum (FFT)
freqs = np.fft.rfftfreq(len(signal), 1 / sample_rate)
spectrum = np.abs(np.fft.rfft(signal))
axes[1].plot(freqs[:2000], spectrum[:2000])
axes[1].set_title('Frequency Domain Spectrum (FFT)')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Magnitude')

plt.tight_layout()
plt.savefig('./images/audio_signal_demo.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'信号长度: {len(signal)} 点, 采样率: {sample_rate} Hz, 时长: {duration}s')
print(f'信号已保存至 ./images/audio_signal_demo.png')

**代码说明与预期结果**：

- **信号构造**：`signal = 0.6*sin(2*200t) + 0.3*sin(2*400t) + 0.1*randn`，即基频200Hz（模拟声带振动基频）+ 谐波400Hz（模拟泛音）+ 随机噪声（模拟环境噪声），三者的振幅比为6:3:1。
- **时域波形（左图）**：展示前50ms（800个采样点）的振幅随时间变化曲线，应呈现以200Hz为主周期的准周期波形，叠加高频抖动（噪声）。
- **频域频谱（右图）**：对信号做FFT后取幅度谱，应在200Hz和400Hz处出现两个明显的峰值，其余频率处幅度很低（噪声底噪），这验证了信号确实由这两个频率分量组成。
- **输出**：打印信号长度为16000点（16000Hz * 1s），图片保存至 `./images/audio_signal_demo.png`。

**为什么这样设计**：通过构造已知频率成分的信号，可以直观验证FFT的正确性——如果频谱在200Hz和400Hz处出现峰值，说明变换正确。这也是理解语音信号处理的第一步：语音在时域看似杂乱，但在频域有清晰的结构。

### 2.4 MFCC 特征提取

**MFCC（Mel-Frequency Cepstral Coefficients）** 是语音识别中最经典的特征，提取流程如下：

```
预加重 → 分帧加窗 → FFT → 梅尔滤波 → 对数运算 → DCT → MFCC系数
```

梅尔频率模拟人耳听觉特性（低频分辨率高，高频分辨率低）：

$$\text{mel}(f) = 2595 \cdot \log_{10}\!\left(1 + \frac{f}{700}\right)$$

下面用纯 Python + NumPy 实现 MFCC 提取，并在 NPU 上加速 DCT 变换。

In [ ]:
import numpy as np

def pre_emphasis(signal, coef=0.97):
    """预加重：提升高频成分，平衡频谱"""
    return np.append(signal[0], signal[1:] - coef * signal[:-1])

def framing(signal, sample_rate, frame_size=0.025, frame_stride=0.010):
    """分帧加窗：25ms 帧长，10ms 帧移，汉明窗"""
    frame_len = int(round(frame_size * sample_rate))
    frame_step = int(round(frame_stride * sample_rate))
    num_frames = 1 + int(np.ceil((len(signal) - frame_len) / frame_step))
    padded = np.append(signal, np.zeros(num_frames * frame_step + frame_len - len(signal)))
    frames = np.lib.stride_tricks.as_strided(
        padded,
        shape=(num_frames, frame_len),
        strides=(padded.strides[0] * frame_step, padded.strides[0])
    )
    frames *= np.hamming(frame_len)
    return frames

def mel_filterbank(num_filters, nfft, sample_rate, low_freq=0, high_freq=None):
    """梅尔滤波器组"""
    high_freq = high_freq or sample_rate // 2
    mel_low = 2595 * np.log10(1 + low_freq / 700)
    mel_high = 2595 * np.log10(1 + high_freq / 700)
    mel_points = np.linspace(mel_low, mel_high, num_filters + 2)
    hz_points = 700 * (10 ** (mel_points / 2595) - 1)
    bin_points = np.floor((nfft + 1) * hz_points / sample_rate).astype(int)
    fbank = np.zeros((num_filters, nfft // 2 + 1))
    for m in range(1, num_filters + 1):
        for k in range(bin_points[m - 1], bin_points[m]):
            fbank[m - 1, k] = (k - bin_points[m - 1]) / (bin_points[m] - bin_points[m - 1])
        for k in range(bin_points[m], bin_points[m + 1]):
            fbank[m - 1, k] = (bin_points[m + 1] - k) / (bin_points[m + 1] - bin_points[m])
    return fbank

def extract_mfcc(signal, sample_rate=16000, num_filters=26, num_cepstra=13, nfft=512):
    """完整 MFCC 提取流程"""
    emphasized = pre_emphasis(signal)
    frames = framing(emphasized, sample_rate)
    mag_frames = np.abs(np.fft.rfft(frames, nfft))
    pow_frames = (1.0 / nfft) * (mag_frames ** 2)
    fbank = mel_filterbank(num_filters, nfft, sample_rate)
    filter_banks = np.dot(pow_frames, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)
    log_fbank = np.log(filter_banks)
    # DCT 变换得到 MFCC
    from scipy.fftpack import dct
    mfcc = dct(log_fbank, type=2, axis=1, norm='ortho')[:, :num_cepstra]
    return mfcc

# 对上面生成的信号提取 MFCC
try:
    mfcc = extract_mfcc(signal, sample_rate=16000)
    print(f'MFCC 特征形状: {mfcc.shape}')
    print(f'  帧数: {mfcc.shape[0]}')
    print(f'  每帧 MFCC 维度: {mfcc.shape[1]}')
    print(f'  前 3 帧 MFCC 系数:')
    for i in range(min(3, len(mfcc))):
        print(f'    帧{i}: {[round(v, 3) for v in mfcc[i]]}')
except ImportError:
    print('scumpy 未安装，跳过 DCT 步骤。安装: pip install scipy')
    print('MFCC 提取流程: 预加重 → 分帧加窗 → FFT → 梅尔滤波 → 对数 → DCT')

**代码说明与预期结果**：

- **预加重**：`x[n] - 0.97*x[n-1]`，一阶高通滤波器，提升高频成分，平衡频谱——语音信号高频能量通常较低，预加重后各频段能量更均匀，有利于后续处理。
- **分帧加窗**：25ms帧长、10ms帧移（帧间重叠15ms），每帧乘以汉明窗减少频谱泄漏。1秒信号约产生100帧。
- **FFT + 梅尔滤波**：对每帧做512点FFT取功率谱，再用26个梅尔三角滤波器滤波，模拟人耳对低频高分辨率、高频低分辨率的特性。
- **对数 + DCT**：取对数压缩动态范围，再做DCT去相关得到13维MFCC系数。
- **预期输出**：MFCC形状约为 `(100, 13)`，即约100帧、每帧13维系数。第0维（C0）能量最大，后续维度值递减。若scipy未安装则打印提示信息。

**为什么用MFCC**：MFCC将语音信号压缩为紧凑的特征向量（13维 vs 原始16000维/帧），既保留了语音的辨识信息，又大幅降低了后续模型的计算量，是传统语音识别（GMM-HMM、DNN-HMM）的标准输入特征。

### 2.5 语音活性检测（VAD）

**VAD（Voice Activity Detection）** 判断音频流中何时存在有效语音，可降低功耗、避免对静默片段进行无谓计算。

最简单的**能量检测法**：计算每帧短时能量，超过阈值判定为语音。

In [ ]:
def vad_energy(signal, sample_rate=16000, frame_size=0.025, frame_stride=0.010, threshold=0.01):
    """基于短时能量的 VAD"""
    frame_len = int(frame_size * sample_rate)
    frame_step = int(frame_stride * sample_rate)
    num_frames = (len(signal) - frame_len) // frame_step + 1
    energies = []
    flags = []
    for i in range(num_frames):
        start = i * frame_step
        frame = signal[start:start + frame_len]
        energy = np.sum(frame ** 2) / frame_len
        energies.append(energy)
        flags.append(1 if energy > threshold else 0)
    return np.array(energies), np.array(flags)

# 构造一段含静音的信号：1秒语音 + 0.5秒静音 + 1秒语音
silence = np.zeros(8000)
mixed = np.concatenate([signal, silence, signal])
energies, flags = vad_energy(mixed, threshold=0.05)

speech_ratio = flags.mean()
print(f'混合信号长度: {len(mixed)} 点 ({len(mixed)/16000:.1f}s)')
print(f'总帧数: {len(flags)}')
print(f'语音帧: {flags.sum()} ({speech_ratio:.1%})')
print(f'静音帧: {(1-flags).sum()} ({1-speech_ratio:.1%})')

**代码说明与预期结果**：

- **信号构造**：将之前生成的1秒语音信号 + 0.5秒全零静音 + 1秒语音信号拼接，总长2.5秒（40000个采样点）。
- **VAD算法**：以25ms帧长、10ms帧移分帧，计算每帧的短时平均能量 `sum(x^2)/N`，超过阈值0.05则标记为语音帧（flag=1），否则为静音帧（flag=0）。
- **预期输出**：总帧数约248帧，语音帧约占66.7%（2秒语音/3秒总长），静音帧约占33.3%（0.5秒静音/3秒总长）。中间0.5秒静音段的帧能量接近0，被正确判为静音。

**为什么有效**：语音段的振幅显著大于静音段（静音段全零，能量为0），因此短时能量能可靠区分两者。在实际系统中，VAD可在静音时关闭后续ASR处理，大幅降低功耗——这对电池供电的端侧设备尤为重要。更复杂的VAD还会结合过零率、频谱平坦度等特征提高鲁棒性。

## 3. 语音识别与合成

### 3.1 语音识别（ASR）

**ASR（Automatic Speech Recognition）** 将语音信号转换为文本，核心流程：

```
前端信号处理 → 声学模型解码 → 语言模型后处理 → 文本输出
```

**性能指标**：
- **词错误率（WER）** = 识别错误词数 / 总词数 × 100%
- **实时率（RTF）** = 处理时间 / 音频时长，RTF < 1 为实时

### 3.2 语音唤醒（KWS）

**KWS（Keyword Spotting）** 让设备从连续语音流中识别特定唤醒词（如"小云小云""Hey Siri"），是语音识别的触发入口。

核心要求：**毫秒级响应、高准确率（>95%）、低功耗、低误唤醒（<1次/24h）**。

### 3.3 语音合成（TTS）

**TTS（Text-to-Speech）** 将文本转换为自然流畅的语音，技术演进：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">阶段</th>
<th style="text-align: left;">技术</th>
<th style="text-align: left;">特点</th>
</tr>
<tr>
<td style="text-align: left;">第一代</td>
<td style="text-align: left;">拼接合成</td>
<td style="text-align: left;">拼接预录音片段，音质好但缺乏灵活性</td>
</tr>
<tr>
<td style="text-align: left;">第二代</td>
<td style="text-align: left;">参数合成</td>
<td style="text-align: left;">数学模型生成参数，灵活但机械感强</td>
</tr>
<tr>
<td style="text-align: left;">第三代</td>
<td style="text-align: left;">神经网络合成</td>
<td style="text-align: left;">深度学习直接生成，自然度接近真人</td>
</tr>
</table>

**表格解读**：TTS技术经历了三代演进。第一代拼接合成将预录音的音节/词组片段按规则拼接，音质好（因为都是真人录音）但无法处理未预录的内容，且韵律不自然。第二代参数合成用数学模型（如HMM）生成声学参数再通过声码器合成波形，灵活性提高但声音有明显机械感。第三代神经网络合成用深度学习模型直接从文本生成波形，自然度接近真人发音，是当前主流方案。

主流模型：**WaveNet**（自回归，音质极佳）、**Tacotron 2**（端到端）、**FastSpeech**（非自回归，速度快 270 倍）、**VITS**（变分自编码器，MOS 4.3）。

**模型对比说明**：WaveNet逐个采样点生成（自回归），音质最好但速度极慢；FastSpeech改为并行生成所有帧（非自回归），速度提升270倍且可控时长；VITS将声学模型和声码器统一为端到端模型，MOS评分4.3（满分5），接近真人水平。在端侧部署中，FastSpeech因速度快、可控性强而更受青睐。

## 4. 自然语言理解（NLU）

NLU 是智能语音系统的"大脑"，将文本转化为机器可执行的结构化指令。

### 4.1 核心处理流程

```
语言结构分析 → 语义分析 → 对话管理 → 响应生成
```

### 4.2 意图识别与槽位填充

**意图识别**：将用户输入映射到预定义意图类别（如设备控制、信息查询、日程管理）。

**槽位填充**：从用户输入中抽取关键参数（如时间、地点、设备、动作）。

示例：

```
用户输入: "明天上午8点提醒我开会"
结构化输出:
  意图: set_reminder
  时间: 明天上午8点
  事件: 开会
```

下面用 Python 实现一个简单的基于规则的意图识别与槽位填充：

In [ ]:
import re

class SimpleNLU:
    """基于规则的简单自然语言理解"""

    INTENT_PATTERNS = {
        'device_control': [
            r'(打开|关闭|开|关).*(灯|空调|电视|窗帘|风扇)',
            r'(灯|空调|电视|窗帘|风扇).*(打开|关闭|开|关)',
        ],
        'set_reminder': [
            r'(提醒|闹钟|定时).*(开会|吃药|起床|上班)',
        ],
        'play_music': [
            r'(播放|放|听).*(音乐|歌|歌曲)',
        ],
        'query_weather': [
            r'(天气|温度|下雨|出太阳)',
        ],
    }

    SLOT_PATTERNS = {
        'action': r'(打开|关闭|开|关|播放|放|听|提醒)',
        'device': r'(灯|空调|电视|窗帘|风扇|音乐|歌|歌曲)',
        'time': r'(明天|今天|后天|上午|下午|\d+点|\d+时)',
        'location': r'(客厅|卧室|厨房|卫生间|阳台)',
        'event': r'(开会|吃药|起床|上班)',
    }

    def parse(self, text):
        intent = self._recognize_intent(text)
        slots = self._fill_slots(text)
        return {'intent': intent, 'slots': slots, 'raw': text}

    def _recognize_intent(self, text):
        for intent, patterns in self.INTENT_PATTERNS.items():
            for pattern in patterns:
                if re.search(pattern, text):
                    return intent
        return 'unknown'

    def _fill_slots(self, text):
        slots = {}
        for slot_name, pattern in self.SLOT_PATTERNS.items():
            match = re.search(pattern, text)
            if match:
                slots[slot_name] = match.group(1)
        return slots

nlu = SimpleNLU()
test_inputs = [
    "打开客厅的灯",
    "把卧室的空调关掉",
    "明天上午8点提醒我开会",
    "播放一首轻音乐",
    "今天天气怎么样",
]

print('=== 意图识别与槽位填充 ===')
for text in test_inputs:
    result = nlu.parse(text)
    print(f'\n输入: "{text}"')
    print(f'  意图: {result["intent"]}')
    print(f'  槽位: {result["slots"]}')

**代码说明与预期结果**：

- **意图识别**：用正则表达式匹配预定义意图模式。例如"打开客厅的灯"匹配 `device_control` 意图（含"打开"+"灯"），"明天上午8点提醒我开会"匹配 `set_reminder` 意图（含"提醒"+"开会"）。
- **槽位填充**：从输入中抽取action（打开/关闭/播放等）、device（灯/空调/音乐等）、time（明天/上午/8点等）、location（客厅/卧室等）、event（开会/起床等）等关键参数。
- **预期输出**：
  - "打开客厅的灯" → 意图: device_control, 槽位: {action: 打开, device: 灯, location: 客厅}
  - "明天上午8点提醒我开会" → 意图: set_reminder, 槽位: {time: 明天, event: 开会}
  - "播放一首轻音乐" → 意图: play_music, 槽位: {action: 播放, device: 音乐}
  - "今天天气怎么样" → 意图: query_weather, 槽位: {time: 今天}

**为什么用规则方法**：基于规则的方法简单直观、无需训练数据、结果可控，适合意图种类有限的封闭场景（如智能家居）。缺点是无法处理未预定义的表达方式，泛化能力有限。实际产品中通常先用规则覆盖高频模式，再用LLM处理复杂或未知意图。

## 5. 大语言模型（LLM）与昇腾部署

### 5.1 LLM 的革命性影响

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">打破"任务烟囱"</td>
<td style="text-align: left;">单一模型处理多种 NLP 任务，无需为每个任务单独训练</td>
</tr>
<tr>
<td style="text-align: left;">深度推理能力</td>
<td style="text-align: left;">自注意力机制捕捉长距离依赖，支持复杂推理</td>
</tr>
<tr>
<td style="text-align: left;">快速适应能力</td>
<td style="text-align: left;">提示工程、RAG、LoRA 微调快速适配新领域</td>
</tr>
</table>

**表格解读**：大语言模型相比传统NLP方法有三大革命性优势。"打破任务烟囱"指传统NLP为每个任务（分类、翻译、摘要）单独训练模型，而LLM用一个模型通过提示即可完成所有任务，大幅降低开发和维护成本。"深度推理能力"源于Transformer的自注意力机制，能捕捉句子中任意距离的依赖关系，支持多步推理。"快速适应能力"通过提示工程（零代码适配）、RAG（外挂知识库）和LoRA微调（仅训练0.1%参数）三种方式快速适配新领域，无需从头训练。

### 5.2 常见大语言模型对比

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">模型</th>
<th style="text-align: left;">规模</th>
<th style="text-align: left;">特点</th>
</tr>
<tr>
<td style="text-align: left;">Qwen2（阿里云）</td>
<td style="text-align: left;">0.5B-72B</td>
<td style="text-align: left;">中文优化，0.5B 适合嵌入式部署</td>
</tr>
<tr>
<td style="text-align: left;">DeepSeek</td>
<td style="text-align: left;">7B-67B</td>
<td style="text-align: left;">中文理解和逻辑推理出色</td>
</tr>
<tr>
<td style="text-align: left;">LLaMA3（Meta）</td>
<td style="text-align: left;">8B-70B</td>
<td style="text-align: left;">开源，8B 版本专为边缘计算优化</td>
</tr>
</table>

**表格解读**：三款主流开源大模型各有特点。Qwen2（阿里云）提供从0.5B到72B的完整规模系列，其中0.5B版本仅约1GB，适合香橙派等嵌入式设备部署，且对中文优化最好。DeepSeek以逻辑推理和数学能力见长，适合需要深度推理的场景。LLaMA3（Meta）是开源生态最完善的模型，8B版本专为边缘计算优化。在本课程中我们选择Qwen1.5-0.5B-Chat，因为它兼顾了中文能力、小体积和开源可商用三大优势。

### 5.3 模型量化技术

将模型参数从高精度（FP32/FP16）转换为低精度整数（INT8/INT4），减少存储和计算开销：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">精度</th>
<th style="text-align: left;">7B 模型体积</th>
<th style="text-align: left;">压缩比</th>
</tr>
<tr>
<td style="text-align: left;">FP16（原版）</td>
<td style="text-align: left;">14.2 GB</td>
<td style="text-align: left;">基准</td>
</tr>
<tr>
<td style="text-align: left;">INT8 量化</td>
<td style="text-align: left;">5.1 GB</td>
<td style="text-align: left;">压缩 75%</td>
</tr>
<tr>
<td style="text-align: left;">INT4 量化</td>
<td style="text-align: left;">2.6 GB</td>
<td style="text-align: left;">压缩 88%</td>
</tr>
</table>

**表格解读**：量化是将模型参数从浮点数（FP16/FP32）转换为低比特整数（INT8/INT4）的技术。以7B模型为例：FP16原版14.2GB，INT8量化后5.1GB（压缩75%），INT4量化后仅2.6GB（压缩88%）。量化原理是将连续的浮点值映射到有限的整数级别，例如INT8将浮点范围映射到[-128, 127]的256个整数。量化后模型不仅存储更小，推理时整数运算也比浮点运算更快。代价是少量精度损失——INT8几乎无损，INT4有轻微性能下降但体积优势显著。对于端侧部署，量化是让大模型塞进有限内存的关键技术。

### 5.4 在昇腾 NPU 上加载 Qwen 大语言模型

下面我们在昇腾 910B3 NPU 上加载 **Qwen1.5-0.5B-Chat** 模型并进行推理，亲手体验大语言模型在昇腾平台上的运行。

> 本环节使用 `torch_npu` 将 PyTorch 模型部署到昇腾 NPU，体验"一行代码切换设备"的便捷。

#### 5.4.1 安装依赖库

> 如果 `transformers` 库未安装，下方代码会自动安装。首次安装可能需要1~2分钟，请耐心等待。

In [ ]:
import subprocess
import sys

def ensure_package(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
        print(f'[OK] {package} 已安装')
    except ImportError:
        print(f'[安装] {package} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f'[完成] {package} 安装成功')

ensure_package('transformers')
ensure_package('huggingface_hub')
print('依赖检查完成!')

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
try:
    import torch_npu
    print(f'昇腾 NPU 是否可用: {torch.npu.is_available()}')
    if torch.npu.is_available():
        print(f'NPU 设备名: {torch.npu.get_device_name(0)}')
        device = torch.device('npu:0')
        torch.npu.set_device(0)
    else:
        device = torch.device('cpu')
except ImportError:
    print('torch_npu 未安装，使用 CPU')
    device = torch.device('cpu')

print(f'PyTorch 版本: {torch.__version__}')
print(f'计算设备: {device}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen1.5-0.5B-Chat'
print(f'加载模型: {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'模型参数量: {total_params / 1e6:.1f}M ({total_params / 1e9:.3f}B)')
print(f'数据类型: float16')
print(f'设备: {device}')

**代码说明与预期结果**：

- **模型加载**：`AutoTokenizer.from_pretrained` 加载分词器（将文本转为token ID序列），`AutoModelForCausalLM.from_pretrained` 加载模型权重。`torch_dtype=torch.float16` 指定半精度加载，相比float32节省一半显存。
- **预期输出**：模型参数量约462M（0.462B），数据类型float16，设备为npu:0（若NPU可用）或cpu。首次运行需从HuggingFace下载模型（约924MB），后续运行从本地缓存加载。
- **为什么用0.5B模型**：Qwen1.5-0.5B-Chat仅5亿参数，FP16下约924MB，可在昇腾910B3（32GB显存）上轻松运行，也适合香橙派（16GB内存）端侧部署。更大的7B模型虽能力更强但需14GB显存，端侧难以承载。

In [ ]:
def chat(question, system_prompt='你是一个友好且有帮助的AI助手，请简洁回答。', max_new_tokens=128):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': question},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', tokenize=True,
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            do_sample=False, top_p=0.9, temperature=0.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
    return response

questions = [
    '什么是人工智能？',
    '用一句话介绍昇腾NPU。',
    '什么是Transformer架构？',
]

for q in questions:
    print(f'\n问: {q}')
    answer = chat(q)
    print(f'答: {answer}')

**代码说明与预期结果**：

- **chat函数**：将system提示和用户问题通过`apply_chat_template`格式化为模型输入，调用`model.generate`生成回复，`do_sample=False`表示贪心解码（每次选概率最高的token，结果确定可复现）。
- **预期输出**：模型会对三个问题给出简洁的中文回答。例如"什么是人工智能？"可能回答"人工智能是计算机科学的一个分支，致力于研究、开发用于模拟、延伸和扩展人类智能的理论、方法、技术及应用系统。"等。回答质量取决于模型的预训练知识。
- **为什么temperature=0.1**：低温使输出分布更尖锐（接近贪心解码），回答更确定、保守，适合知识问答场景。高温（如0.8）则增加随机性，适合创意写作。

> **注意**：如果上方模型加载单元格因 `transformers` 未安装而报错，请先运行「5.4.1 安装依赖库」单元格安装依赖，然后重新运行模型加载单元格，再运行本单元格。

### 5.5 NPU 加速感受：张量矩阵乘法

用一个大矩阵乘法任务，对比 CPU 与昇腾 NPU 的计算速度，直观感受 NPU 的并行加速能力。

In [ ]:
import time

N = 4000
a = torch.randn(N, N, dtype=torch.float32)
b = torch.randn(N, N, dtype=torch.float32)
print(f'矩阵大小: {N}x{N}, 约 {a.numel() * 4 / 1024**2:.0f} MB')

# CPU 计时
start = time.time()
c_cpu = torch.matmul(a, b)
cpu_time = time.time() - start
print(f'CPU 矩阵乘法耗时: {cpu_time*1000:.1f} ms')

# NPU 计时
if str(device) != 'cpu':
    a_npu = a.npu()
    b_npu = b.npu()
    torch.npu.synchronize()
    start = time.time()
    c_npu = torch.matmul(a_npu, b_npu)
    torch.npu.synchronize()
    npu_time = time.time() - start
    print(f'NPU 矩阵乘法耗时: {npu_time*1000:.1f} ms')
    print(f'加速比: {cpu_time/npu_time:.1f}x')

    # 验证结果一致性
    match = torch.allclose(c_npu.cpu(), c_cpu, rtol=1e-3, atol=1e-1)
    print(f'NPU 与 CPU 结果一致: {match}')
else:
    print('NPU 不可用，跳过 NPU 测试')

**代码说明与预期结果**：

- **测试任务**：两个4000x4000的随机float32矩阵相乘，数据量约61MB，计算量约1280亿次浮点乘加运算。
- **CPU计时**：在CPU上执行`torch.matmul`，预期耗时数百毫秒至数秒（取决于CPU核心数）。
- **NPU计时**：将矩阵搬到NPU后执行，需`torch.npu.synchronize()`确保异步计算完成再计时。预期耗时数十毫秒，加速比可达10~50倍。
- **结果验证**：`torch.allclose`比较NPU和CPU结果，允许相对误差1e-3、绝对误差1e-1。由于浮点运算顺序不同，结果不会完全一致但应在容差范围内。
- **为什么NPU快**：昇腾NPU内置3D Cube矩阵计算单元，单周期可完成大量乘加运算，且拥有高带宽HBM显存，远超CPU的标量/向量计算能力。`synchronize`是必要的因为NPU计算是异步的——调用返回时计算可能未完成，不同步会导致计时不准。

## 6. 昇腾平台语音处理实践

### 6.1 昇腾开发板音频接口

昇腾香橙派开发板集成完整音频处理电路，通过**音频三通道接口**实现音频输入输出：
- 2 个输出通道：左右声道音频输出
- 1 个输入通道：麦克风音频采集

音频输入/输出方案：
- **3.5mm 耳机/麦克风接口**：模拟音频
- **HDMI 音频**：数字音频输出
- **板载扬声器/麦克风**：内置设备
- **USB 声卡**：扩展音频设备

### 6.2 音频测试命令（香橙派）

在香橙派开发板上，使用 `sample_audio` 程序测试音频：

```bash
# 音频播放（2 = 3.5mm接口）
./sample_audio play 2 qzgy_48k_16_mono_30s.pcm

# 音频采集
./sample_audio capture test.pcm

# 回放录音验证
./sample_audio play 2 test.pcm
```

### 6.3 Wav2word 语音识别案例

Wav2word 是端到端自动语音识别模型，将语音信号直接转换为文本：

```
Input(1600×200 梅尔谱图) → Conv+Pool×5 → Flatten → FC×2 → Softmax(1424类)
```

部署到昇腾的流程：
1. 创建网络结构，加载预训练权重
2. 冻结计算图，生成 PB 文件
3. 使用 **ATC 工具**转换为 OM 格式

```bash
atc --model='./Wav2word.pb' \\
    --output_type=FP32 \\
    --output='./Wav2word_om' \\
    --framework=3 \\
    --soc_version=Ascend310
```

> ATC（Ascend Tensor Compiler）是昇腾模型转换工具，支持 TensorFlow、Caffe、ONNX 等框架模型转换为昇腾 OM 格式，充分发挥 NPU 算力优势。

## 7. 性能调优与参数配置

大语言模型推理的关键参数：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">典型值</th>
<th style="text-align: left;">作用</th>
</tr>
<tr>
<td style="text-align: left;">temperature</td>
<td style="text-align: left;">0.1</td>
<td style="text-align: left;">低温使输出更确定、保守</td>
</tr>
<tr>
<td style="text-align: left;">top_p</td>
<td style="text-align: left;">0.9</td>
<td style="text-align: left;">核采样，平衡多样性和质量</td>
</tr>
<tr>
<td style="text-align: left;">max_new_tokens</td>
<td style="text-align: left;">1024</td>
<td style="text-align: left;">限制生成文本最大长度</td>
</tr>
<tr>
<td style="text-align: left;">do_sample</td>
<td style="text-align: left;">True</td>
<td style="text-align: left;">启用采样解码</td>
</tr>
</table>

**表格解读**：四个参数控制LLM的生成行为。`temperature`（温度）控制输出随机性——值越低输出越确定保守（适合知识问答），值越高越随机多样（适合创意写作），0.1是低温保守设置。`top_p`（核采样）只从累积概率超过p的最小token集合中采样，0.9表示排除概率极低的长尾token，平衡多样性和质量。`max_new_tokens`限制生成长度防止无限输出。`do_sample`设为True启用采样解码（结合temperature和top_p），设为False则纯贪心解码（每次选最高概率token，结果确定可复现）。

### 优化策略

- **半精度推理（FP16）**：减少内存占用，5 亿参数模型可在边缘设备流畅运行
- **流式响应**：token 级别逐步生成，用户实时观察生成过程
- **上下文管理**：维护多轮对话历史，连贯回应
- **模型量化**：INT8/INT4 量化，压缩 75%~88%

---

## 小结

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">技术模块</th>
<th style="text-align: left;">核心技术</th>
</tr>
<tr>
<td style="text-align: left;">语音采集与前处理</td>
<td style="text-align: left;">麦克风阵列、ADC、降噪、VAD、AEC</td>
</tr>
<tr>
<td style="text-align: left;">语音识别与合成</td>
<td style="text-align: left;">ASR、MFCC、KWS、TTS、端到端模型</td>
</tr>
<tr>
<td style="text-align: left;">自然语言理解</td>
<td style="text-align: left;">意图识别、槽位填充、对话管理、LLM</td>
</tr>
<tr>
<td style="text-align: left;">边缘部署实践</td>
<td style="text-align: left;">昇腾芯片、CANN、MindSpore/PyTorch、模型量化</td>
</tr>
</table>

**表格解读**：本表格总结了本课程涉及的四大技术模块及其核心技术点。语音采集与前处理是系统入口，关键技术围绕信号采集和质量提升；语音识别与合成是核心转换环节，从传统的MFCC+HMM发展到端到端神经网络模型；自然语言理解是智能大脑，从规则匹配发展到LLM统一处理；边缘部署实践是落地关键，通过昇腾芯片+CANN+量化技术让大模型在端侧设备上高效运行。四大模块构成完整的智能语音技术栈，在本课程中通过理论讲解+代码实践的方式逐一学习。

## 课后练习

请根据本节课程学习内容完成以下题目进行自测。

**第1题**（单选题）智能语音系统的三大核心环节不包括以下哪项？

- A. 语音采集与前处理
- B. 语音识别与合成
- C. 自然语言理解
- D. 图像识别与处理


In [ ]:
q1 = ''  # 填入你的选项，如 'D'，修改后务必运行本单元格（Shift+Enter）
print(f'第{1}题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）根据奈奎斯特采样定理，人耳可听频率上限约 20 kHz，CD 音质采用的采样率是？

- A. 8 kHz
- B. 16 kHz
- C. 44.1 kHz
- D. 96 kHz


In [ ]:
q2 = ''
print(f'第{2}题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）MFCC 特征提取的正确流程顺序是？

- A. 预加重 → 分帧加窗 → FFT → 梅尔滤波 → 对数运算 → DCT
- B. FFT → 预加重 → 分帧加窗 → DCT → 梅尔滤波 → 对数运算
- C. 分帧加窗 → DCT → FFT → 预加重 → 梅尔滤波 → 对数运算
- D. 梅尔滤波 → FFT → 预加重 → 分帧加窗 → 对数运算 → DCT


In [ ]:
q3 = ''
print(f'第{3}题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）语音唤醒（KWS）的核心要求不包括？

- A. 毫秒级响应
- B. 高准确率（>95%）
- C. 低功耗
- D. 支持任意语言翻译


In [ ]:
q4 = ''
print(f'第{4}题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）以下哪个 TTS 模型是非自回归的，推理速度提升 270 倍？

- A. WaveNet
- B. Tacotron 2
- C. FastSpeech
- D. VITS


In [ ]:
q5 = ''
print(f'第{5}题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**第6题**（单选题）用户输入"明天上午8点提醒我开会"，意图识别的结果应该是？

- A. device_control
- B. set_reminder
- C. play_music
- D. query_weather


In [ ]:
q6 = ''
print(f'第{6}题答案已记录：{q6}' if q6 else '⚠️ 请填入答案并运行本单元格')

**第7题**（单选题）大语言模型的革命性优势"打破任务烟囱"指的是？

- A. 用单一模型处理多种 NLP 任务
- B. 模型参数量达到 100B 以上
- C. 支持图像、音频、视频等多模态
- D. 训练数据量达到 10T tokens


In [ ]:
q7 = ''
print(f'第{7}题答案已记录：{q7}' if q7 else '⚠️ 请填入答案并运行本单元格')

**第8题**（单选题）INT8 量化对 7B 模型的压缩比约为？

- A. 50%
- B. 75%
- C. 88%
- D. 95%


In [ ]:
q8 = ''
print(f'第{8}题答案已记录：{q8}' if q8 else '⚠️ 请填入答案并运行本单元格')

**第9题**（单选题）ATC 工具的作用是？

- A. 训练大语言模型
- B. 将模型转换为昇腾 OM 格式
- C. 进行语音特征提取
- D. 管理对话上下文


In [ ]:
q9 = ''
print(f'第{9}题答案已记录：{q9}' if q9 else '⚠️ 请填入答案并运行本单元格')

**第10题**（单选题）Qwen1.5-0.5B-Chat 模型的参数量约为？

- A. 5 亿（0.5B）
- B. 70 亿（7B）
- C. 720 亿（72B）
- D. 1000 亿（100B）


In [ ]:
q10 = ''
print(f'第{10}题答案已记录：{q10}' if q10 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd().parent / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_01 import grade
grade(globals())

## 参考资料

- [MindSpore 官方网站](https://www.mindspore.cn/)
- [MindNLP 代码仓](https://github.com/mindspore-lab/mindnlp)
- [Qwen 模型](https://huggingface.co/Qwen/Qwen1.5-0.5B-Chat)
- [昇腾 CANN 文档](https://www.hiascend.com/document)
- [香橙派 AI Pro 官方文档](http://www.orangepi.cn/)